In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
import pandas as pd 

df = pd.read_csv(r"C:\Users\manas\Downloads\ALONE\New folder\new\PAIMANA\ml-pipeline\data\processed\merged_projects.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()
df.columns
df.describe(include="all")
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)].head(20)

In [ ]:
df[df.duplicated(keep=False)][["project_code" ,"project_name" ,"status"]].head(10)

In [ ]:
df[df.duplicated()].head(10)

In [ ]:
df[df.duplicated()].shape

In [ ]:
df[df.duplicated()].status.value_counts()

In [ ]:
df[df.duplicated()][["project_code","project_name","date_of_approval" ,"start_date" ,"target_doc","status"]].head(20)

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

In [ ]:
date_columns = [
    "date_of_approval",
    "start_date",
    "target_doc",
    "revised_doc",
    "actual_doc"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [ ]:
df[date_columns].dtypes

In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0].sort_values(ascending=False)

In [ ]:
df.groupby("status")[[
    "date_of_approval",
    "start_date",
    "target_doc",
    "revised_doc",
    "actual_doc",
    "revised_cost_cr",
    "ministry"
]].apply(lambda x: x.isnull().sum())

In [ ]:
df[df["state"].isnull()][
    ["project_code", "project_name", "agency", "state"]
]

In [ ]:
df[df["cumulative expenditure in rs. crore"] < 0][
    ["project_code", "project_name",
     "cumulative expenditure in rs. crore", "status"]
]

In [ ]:
# Data quality issue:
# 2 Ongoing projects have negative cumulative expenditure.
# Values: -4.88 and -0.08 crore.
# As per data dictionary, these are data errors.
# Correction requires source lookup; no automatic imputation applied.

In [ ]:
df[df["revised_cost_cr"] < df["original_cost_cr"]][
    ["project_code", "project_name",
     "original_cost_cr", "revised_cost_cr",
     "cost_overrun_abs_cr", "cost_overrun_pct"]
].head(10)

In [ ]:
# Data quality observation:
# 2,549 projects have revised_cost_cr < original_cost_cr.
# Negative cost overrun is not necessarily an error and may represent
# de-scoping or cost savings. Values are preserved as-is.

In [ ]:
target_columns = [
    "cost_overrun_abs_cr",
    "cost_overrun_pct",
    "delay_actual_days",
    "delay_proxy_days"
]

df[target_columns].info()

In [ ]:
df[target_columns].notnull().sum()

In [ ]:
target_columns = [
    "cost_overrun_abs_cr",
    "cost_overrun_pct",
    "delay_actual_days",
    "delay_proxy_days"
]

df[target_columns].info()

In [ ]:
df[target_columns].notnull().sum()

In [ ]:
leakage_columns = [
    "revised_cost_cr",
    "revised_doc",
    "actual_doc",
    "cumulative expenditure in rs. crore"
]

df[leakage_columns].head()

In [ ]:
df[leakage_columns].isnull().sum()

In [ ]:
leakage_columns = [
    "revised_cost_cr",
    "revised_doc",
    "actual_doc",
    "cumulative expenditure in rs. crore"
]


In [ ]:
df[leakage_columns].isnull().sum()

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes

In [ ]:
df["delay_actual_days"].value_counts(dropna=False).head(20)

In [ ]:
df["delay_proxy_days"].value_counts(dropna=False).head(20)

In [ ]:
df["delay_actual_days"] = (
    df["delay_actual_days"]
    .str.replace(" days", "", regex=False)
    .astype(float)
)

In [ ]:
df["delay_actual_days"].dtype

In [ ]:
df["delay_proxy_days"] = (
    df["delay_proxy_days"]
    .str.replace(" days", "", regex=False)
    .astype(float)
)

In [ ]:
df[["delay_actual_days","delay_proxy_days"]].dtypes

In [ ]:
numeric_columns = [
    "original_cost_cr",
    "revised_cost_cr",
    "cumulative expenditure in rs. crore",
    "physical progress (in percentage)",
    "cost_overrun_abs_cr",
    "cost_overrun_pct",
    "delay_actual_days",
    "delay_proxy_days"
]

df[numeric_columns].describe().T

In [ ]:
missing_flag_columns = [
    "date_of_approval_is_missing",
    "start_date_is_missing",
    "actual_doc_is_missing",
    "target_doc_is_missing",
    "revised_doc_is_missing",
    "revised_cost_cr_is_missing",
    "ministry_is_missing"
]

df[missing_flag_columns].sum()

In [ ]:
quality_flag_columns = [
    "negative_expenditure_flag",
    "negative_cost_overrun_flag",
    "expenditure_exceeds_revised_cost",
    "high_spend_low_progress_flag"
]

df[quality_flag_columns].sum()

In [ ]:
quality_flag_columns = [
    "negative_expenditure_flag",
    "negative_cost_overrun_flag",
    "expenditure_exceeds_revised_cost",
    "high_spend_low_progress_flag"
]

df[quality_flag_columns].sum()

In [ ]:
quality_flag_columns = [
    "negative_expenditure_flag",
    "negative_cost_overrun_flag",
    "expenditure_exceeds_revised_cost",
    "high_spend_low_progress_flag"
]

df[quality_flag_columns].sum()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nData types:")
print(df.dtypes)

In [ ]:
df.head()

In [ ]:
output_path = "../data/processed/merged_projects_cleaned.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print(output_path)

In [ ]:
import os
print(os.path.exists(output_path))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Scikit-learn imported successfully!")

In [ ]:
# Target
target = "delay_proxy_days"

# Columns that must NOT be used as features
excluded_columns = [
    "delay_proxy_days",
    "delay_actual_days",
    "revised_doc",
    "actual_doc",
    "revised_cost_cr",
    "cumulative expenditure in rs. crore",
    "cost_overrun_abs_cr",
    "cost_overrun_pct",
    "status",
    "project_code",
    "project_name"
]

# Features
X = df.drop(columns=excluded_columns)

# Target
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Numerical columns
numerical_columns = X.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()

# Preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocessing for numerical columns
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

print("Preprocessing pipeline created successfully!")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# Create Random Forest pipeline
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# Keep only rows where the target is available
model_data = df[df["delay_proxy_days"].notna()].copy()

# Features and target
X = model_data.drop(columns=excluded_columns)
y = model_data["delay_proxy_days"]

print("Model data shape:", model_data.shape)
print("Target missing:", y.isna().sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# Make predictions on test data
y_pred = rf_model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Model Performance")
print("------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
results = pd.DataFrame({
    "Actual Delay (days)": y_test.values,
    "Predicted Delay (days)": y_pred
})

results.head(10)

In [ ]:
# Get feature names after preprocessing
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

# Get feature importance
importances = rf_model.named_steps["model"].feature_importances_

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feature_importance.head(15)

In [ ]:
model_results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [mae, rmse, r2]
})

model_results

In [ ]:
top_features =feature_importance.head(15)
top_features

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.xlabel("Actual Delay (days)")
plt.ylabel("Predicted Delay (days)")
plt.title("Actual vs Predicted Delay - Random Forest")
plt.show()

In [ ]:
import joblib

joblib.dump(rf_model, "../data/processed/random_forest_delay_model.pkl")

print("Model saved successfully!")

In [ ]:
import os

print(os.path.exists("../data/processed/merged_projects_cleaned.csv"))
print(os.path.exists("../data/processed/random_forest_delay_model.pkl"))